# 실기 대비
# 실전 문제풀이
# set 5

## 1) 데이터 및 시나리오

### 신용카드 고객정보 분석

> 삼성전자의 고객 마케팅에 활용하기 위하여 삼성 카드 측에서 비식별화된 고객 데이터를 협조 받아 통합 분석을 하기 위한 선제 분석을 하고자 한다.

※ 분석 수행 전 `기한 내 최소 지불 금액(MINIMUM_PAYMENTS)`의 결측 값(Null)을 각 컬럼의 평균값으로 대체하시오.

전처리 수행 결과를 `base` 객체로 지정하고 다음 문항에서 해당 객체를 기반으로 문제를 풀이하시오.

### 데이터 개요

| 파일명 | 행 | 열 | 인코딩 |
|---|---:|---:|---|
| `card_cust.csv` | 1000 | 18 | UTF-8 |

## 1) 데이터 및 시나리오

### 변수 상세

| 변수명 | 유형 | 설명 |
|---|---|---|
| `CUST_ID` | int | 고객 ID |
| `BALANCE` | float | 연간 평균 잔고액 |
| `BALANCE_FREQUENCY` | float | 연중 잔고액 갱신 개월 수 비율 `(0~1 사이값)` |
| `PURCHASES` | float | 구매 총액 |
| `ONEOFF_PURCHASES` | float | 일시불 구매 총액 |
| `INSTALLMENTS_PURCHASES` | float | 할부 구매 총액 |
| `CASH_ADVANCE` | float | 현금서비스 구매 총액 |
| `PURCHASES_FREQUENCY` | float | 연중 구매 개월 수 비율 `(0~1 사이값)` |
| `ONEOFF_PURCHASES_FREQUENCY` | float | 연중 일시불 구매 개월 수 비율 `(0~1 사이값)` |
| `PURCHASES_INSTALLMENTS_FREQUENCY` | float | 연중 할부 구매 개월 수 비율 `(0~1 사이값)` |
| `CASH_ADVANCE_FREQUENCY` | float | 연중 현금서비스 구매 개월 수 비율 |
| `CASH_ADVANCE_TRX` | int | 현금 서비스 구매 횟수 |
| `PURCHASES_TRX` | int | 구매 횟수 |
| `CREDIT_LIMIT` | int | 신용카드 한도 |
| `PAYMENTS` | float | 지불 총액 |
| `MINIMUM_PAYMENTS` | float | 기한 내 최소 지불 금액 |
| `PRC_FULL_PAYMENT` | float | 연중 기한 내 전액 지불 개월 수 비율 `(0~1 사이값)` |
| `TENURE` | float | 신용카드 서비스 이용기간 |

## 2) 문제

### 필요 라이브러리 함수 및 클래스 목록

| 목록 |
|---|
| `from sklearn.preprocessing import StandardScaler` |
| `from sklearn.cluster import KMeans` |
| `from sklearn.metrics import silhouette_score` |
| `from sklearn.tree import DecisionTreeRegressor` |

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.tree import DecisionTreeRegressor

In [7]:
df = pd.read_csv('../../dataset/card_cust.csv')
#SCDI
display(df.shape)
display(df.columns)
display(df.dtypes)
display(df.isna().sum())
df['MINIMUM_PAYMENTS'] = df['MINIMUM_PAYMENTS'].fillna(df['MINIMUM_PAYMENTS'].mean())
display(df.isna().sum())
base = df.copy()

(1000, 18)

Index(['CUST_ID', 'BALANCE', 'BALANCE_FREQUENCY', 'PURCHASES',
       'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE',
       'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY',
       'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY',
       'CASH_ADVANCE_TRX', 'PURCHASES_TRX', 'CREDIT_LIMIT', 'PAYMENTS',
       'MINIMUM_PAYMENTS', 'PRC_FULL_PAYMENT', 'TENURE'],
      dtype='object')

CUST_ID                               int64
BALANCE                             float64
BALANCE_FREQUENCY                   float64
PURCHASES                           float64
ONEOFF_PURCHASES                    float64
INSTALLMENTS_PURCHASES              float64
CASH_ADVANCE                        float64
PURCHASES_FREQUENCY                 float64
ONEOFF_PURCHASES_FREQUENCY          float64
PURCHASES_INSTALLMENTS_FREQUENCY    float64
CASH_ADVANCE_FREQUENCY              float64
CASH_ADVANCE_TRX                    float64
PURCHASES_TRX                       float64
CREDIT_LIMIT                        float64
PAYMENTS                            float64
MINIMUM_PAYMENTS                    float64
PRC_FULL_PAYMENT                    float64
TENURE                              float64
dtype: object

CUST_ID                              0
BALANCE                              0
BALANCE_FREQUENCY                    0
PURCHASES                            0
ONEOFF_PURCHASES                     0
INSTALLMENTS_PURCHASES               0
CASH_ADVANCE                         0
PURCHASES_FREQUENCY                  0
ONEOFF_PURCHASES_FREQUENCY           0
PURCHASES_INSTALLMENTS_FREQUENCY     0
CASH_ADVANCE_FREQUENCY               0
CASH_ADVANCE_TRX                     0
PURCHASES_TRX                        0
CREDIT_LIMIT                         0
PAYMENTS                             0
MINIMUM_PAYMENTS                    74
PRC_FULL_PAYMENT                     0
TENURE                               0
dtype: int64

CUST_ID                             0
BALANCE                             0
BALANCE_FREQUENCY                   0
PURCHASES                           0
ONEOFF_PURCHASES                    0
INSTALLMENTS_PURCHASES              0
CASH_ADVANCE                        0
PURCHASES_FREQUENCY                 0
ONEOFF_PURCHASES_FREQUENCY          0
PURCHASES_INSTALLMENTS_FREQUENCY    0
CASH_ADVANCE_FREQUENCY              0
CASH_ADVANCE_TRX                    0
PURCHASES_TRX                       0
CREDIT_LIMIT                        0
PAYMENTS                            0
MINIMUM_PAYMENTS                    0
PRC_FULL_PAYMENT                    0
TENURE                              0
dtype: int64

### Q01.

`base`를 사용하여 연간 평균 잔고액과 신용카드 서비스 이용기간 간의 관계를 파악하여, 추후 고객의 신용카드 한도 조정에 근거 자료로 활용하고자 한다.

연간 평균 잔고액(`BALANCE`)이 많을수록, 그리고 신용카드 서비스 이용기간(`TENURE`)이 길수록 신용카드 한도(`CREDIT_LIMIT`) 역시 높을 것으로 예상해볼 수 있다. 신용 카드 서비스 이용기간(`TENURE`) 별로 연간 평균 잔고액(`BALANCE`)과 신용카드 한도(`CREDIT_LIMIT`) 간 피어슨(Pearson) 상관 분석을 실시하고, 이 중 가장 큰 상관계수를 구하시오.

※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [ ]:
df_q1 = base.copy()
cols_q1 = ['BALANCE', 'TENURE', 'CREDIT_LIMIT']
def corr_(df) :
    df_corr = df[['BALANCE', 'CREDIT_LIMIT']].corr()
    np.fill_diagonal(df_corr.values, 0)
    return df_corr
df_q1_corr = df_q1[cols_q1].groupby('TENURE').apply(corr_)
display(df_q1_corr, df_q1_corr.max())
display(round(df_q1_corr.max().max(),2))

BALANCE  CREDIT_LIMIT
TENURE                                     
6.0    BALANCE       0.000000      0.868056
       CREDIT_LIMIT  0.868056      0.000000
7.0    BALANCE       0.000000      0.948405
       CREDIT_LIMIT  0.948405      0.000000
8.0    BALANCE       0.000000      0.820696
       CREDIT_LIMIT  0.820696      0.000000
9.0    BALANCE       0.000000      0.085474
       CREDIT_LIMIT  0.085474      0.000000
10.0   BALANCE       0.000000      0.291482
       CREDIT_LIMIT  0.291482      0.000000
11.0   BALANCE       0.000000      0.380360
       CREDIT_LIMIT  0.380360      0.000000
12.0   BALANCE       0.000000      0.460833
       CREDIT_LIMIT  0.460833      0.000000

BALANCE         0.948405
CREDIT_LIMIT    0.948405
dtype: float64

0.95

,BALANCE,TENURE,CREDIT_LIMIT
BALANCE,0.000000,0.002592,0.467646
TENURE,0.002592,0.000000,0.059645
CREDIT_LIMIT,0.467646,0.059645,0.000000


BALANCE         0.467646
TENURE          0.059645
CREDIT_LIMIT    0.467646
dtype: float64

0.46764597335688984

### Q02.

`base`를 사용하여 전략을 수립하기 위해 고객 세분화를 수행하고자 한다.  
일시불 구매 금액이 높은 고객군을 도출하기 위해 다음 단계에 따라 분석을 수행하고 질문에 답하시오.

#### 단계 1

`고객 ID`를 제외한 모든 변수 17개에 대해 Z-score 표준화(Standardization) 한다.

#### 단계 2

표준화된 변수들에 대해 K-means 군집 분석을 수행한다.

이 때, 군집 수는 2~5개 중 K-means Silhouette를 통해 구한 최적의 K로 설정한다.

#### 단계 3

단계 2에서 도출한 각 군집 별로 `일시불 구매 총액`의 평균을 계산한다.

**군집 별 일시불 구매 총액(`ONEOFF_PURCHASES`)의 평균 중 가장 큰 값은 얼마인가?**

※ 정규화를 실시하지 않은 일시불 구매 총액 데이터를 기준으로 평균을 산출하시오.  
※ seed는 `1234`로 설정하시오.  
※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [48]:
df_q2 = base.copy()
# 모든변수가 float라서 가능하다. --> 항상 확인하기
#단계 1
df_q2_1 = df_q2.drop(columns='CUST_ID').copy()
display(df_q2_1)
scaler = StandardScaler()
df_q2_n = scaler.fit_transform(df_q2_1)
#display(arr_n)
#df_q2_n = pd.DataFrame(arr_n, columns=df_q2_1.columns)

#단계 2
dict_q2 = {}
dict_q2_label = {}
for k in [2,3,4,5] :
    model = KMeans(n_clusters = k, random_state=1234)
    model.fit(df_q2_n)
    pred = model.predict(df_q2_n)
    dict_q2[k] = silhouette_score(df_q2_n, pred)
    dict_q2_label[k] = pred

ser_q2 = pd.Series(dict_q2)
display(ser_q2.idxmax(), ser_q2.max(), ser_q2)
df_q2_1['cluster'] = dict_q2_label[ser_q2.idxmax()]
display(df_q2_1['cluster'].value_counts())
df_q2_gb = df_q2_1.groupby('cluster')['ONEOFF_PURCHASES'].mean()
display(df_q2_gb, round(df_q2_gb.max(),2))

,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,40.900749,0.818182,95.40,0.00,95.40,0.000000,0.166667,0.000000,0.083333,0.000000,0.0,2.0,1000.0,201.802084,139.509787,0.000000,12.0
1,3202.467416,0.909091,0.00,0.00,0.00,6442.945483,0.000000,0.000000,0.000000,0.250000,4.0,0.0,7000.0,4103.032597,1072.340217,0.222222,12.0
2,2495.148862,1.000000,773.17,773.17,0.00,0.000000,1.000000,1.000000,0.000000,0.000000,0.0,12.0,7500.0,622.066742,627.284787,0.000000,12.0
3,1666.670542,0.636364,1499.00,1499.00,0.00,205.788017,0.083333,0.083333,0.000000,0.083333,1.0,1.0,7500.0,0.000000,1297.116322,0.000000,12.0
4,817.714335,1.000000,16.00,16.00,0.00,0.000000,0.083333,0.083333,0.000000,0.000000,0.0,1.0,1200.0,678.334763,244.791237,0.000000,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1250.394614,0.909091,443.99,443.99,0.00,0.000000,0.272727,0.272727,0.000000,0.000000,0.0,3.0,3500.0,273.823646,259.715939,0.000000,11.0
996,9.503968,1.000000,96.62,0.00,96.62,0.000000,1.000000,0.000000,1.000000,0.000000,0.0,19.0,4500.0,1086.932525,92.217936,0.250000,12.0
997,2285.068731,1.000000,0.00,0.00,0.00,1173.310874,0.000000,0.000000,0.000000,0.166667,5.0,0.0,2500.0,381.672065,1003.265207,0.000000,12.0
998,2928.756699,1.000000,160.92,0.00,160.92,319.931964,1.000000,0.000000,1.000000,0.333333,5.0,12.0,3000.0,1142.847203,1098.479834,0.000000,12.0


2

0.30752815304560793

2    0.307528
3    0.196361
4    0.207151
5    0.192741
dtype: float64

0    802
1    198
Name: cluster, dtype: int64

cluster
0     340.230998
1    3946.187525
Name: ONEOFF_PURCHASES, dtype: float64

3946.19

### Q03.

`base`를 사용하여 일시불 구매 총액(`ONEOFF_PURCHASES`) 예측 모델을 Target Marketing에 활용하고자 한다. 다음 단계에 따라 분석을 수행하고 질문에 답하시오.

#### 단계 1

`고객 ID(CUST_ID)`가 4의 배수가 아닌 데이터를 Train Set으로, 4의 배수인 데이터를 Test Set으로 분할한다.

#### 단계 2

Train Set으로 아래 조건에 따라 의사결정나무 회귀모델을 학습한다.

- 독립 변수 총 16개: `고객 ID`, `일시불 구매 총액`을 제외한 모든 변수
- 종속 변수: `일시불 구매 총액`

#### 단계 3

생성된 모델을 Test Set에 적용하여 `일시불 구매 총액`을 예측한다.

**단계 3에서 얻은 예측 결과를 평가하기 위해, 아래 정의된 Measure B를 계산한 값은?**

$$
B = \left( \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y_i})^2 \right)^{\frac{1}{2}}
$$

- $\hat{y_i}$: 예측값
- $y_i$: 실제값

※ seed는 `1234`로 설정하시오.  
※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [52]:
df_q3 = base.copy()
display(df_q3['CUST_ID'].value_counts())
train = df_q3.loc[df_q3['CUST_ID']%4 != 0, ].copy()
test = df_q3.loc[df_q3['CUST_ID']%4 == 0, ].copy()

train_X = train.drop(columns=['CUST_ID','ONEOFF_PURCHASES']).copy()
train_y = train['ONEOFF_PURCHASES'].copy()

test_X = test.drop(columns=['CUST_ID','ONEOFF_PURCHASES']).copy()
test_y = test['ONEOFF_PURCHASES'].copy()

model = DecisionTreeRegressor(random_state = 1234)
model.fit(train_X, train_y)
pred_y = model.predict(test_X)

B = ((test_y - pred_y)**2).mean()**0.5
round(B,2)

10001    1
10698    1
10684    1
10685    1
10686    1
        ..
10349    1
10350    1
10351    1
10352    1
11033    1
Name: CUST_ID, Length: 1000, dtype: int64

1039.19